In [17]:
import argparse
import math
import random
import os
from collections import OrderedDict
from typing import Tuple

import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# -----------------------
# Reproducibility helpers
# -----------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = False
    cudnn.benchmark = True  # faster on GPUs for conv nets

# -----------------------
# Data
# -----------------------
def get_cifar10_loaders(batch_size: int = 128, num_workers: int = 4) -> Tuple[DataLoader, DataLoader]:
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_ds = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_tf)
    test_ds  = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader

In [2]:

# -----------------------
# Train / Eval
# -----------------------
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, correct, running_loss = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss()
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

In [18]:
def train_and_evaluate(model_name, num_classes=10):
  if model_name == 'alexnet':
    model = AlexNetCIFAR()
  else:
    model = DepthwiseAlexNetCIFAR()

  model.to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

  best_acc = 0.0
  for epoch in range(1, epochs + 1):
      train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
      val_loss,  val_acc  = evaluate(model, test_loader, device)

      if val_acc > best_acc:
          best_acc = val_acc
          os.makedirs("checkpoints", exist_ok=True)
          torch.save({"model": model.state_dict(),
                      "epoch": epoch,
                      "acc": best_acc
                      },
                      f"checkpoints/{model_name}_best.pt")

      print(f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss {train_loss:.4f} Acc {train_acc*100:.2f}% | "
            f"Val Loss {val_loss:.4f} Acc {val_acc*100:.2f}% | "
            f"Best Val Acc {best_acc*100:.2f}%")

  # Final evaluation
  test_loss, test_acc = evaluate(model, test_loader, device)
  print(f"\nFinal {model_name.upper()} Test Accuracy: {test_acc*100:.2f}% (loss {test_loss:.4f})")
  return test_acc


# -----------------------
# Forward pass speed
# -----------------------

def benchmark_forward_pass(
    model,
    input_size=(1, 3, 32, 32),
    num_warmup=10,
    num_iters=100
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    x = torch.randn(*input_size, device=device) # Some random input

    # Warmup (to avoid initial overheads)
    with torch.no_grad():
        for _ in range(num_warmup):
            _ = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()

    # Timed runs
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(num_iters):
            _ = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
    end = time.perf_counter()

    avg_time_s = (end - start) / num_iters
    avg_time_ms = avg_time_s * 1000
    print(f"Device: {device}")
    print(f"Input size: {input_size}")
    print(f"Average forward pass time: {avg_time_ms:.4f} ms over {num_iters} iters")
    return avg_time_ms

In [14]:
class AlexNetCIFAR(nn.Module):
    """
    AlexNet adapted for 32x32 inputs:
    - Use 3x3 convs (stride 1) instead of 11x11/5x5
    - Slightly reduced channels to fit CIFAR-10 scale
    """
    def __init__(self, num_classes: int = 10, dropout: float = 0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=1, padding=1),  # 32x32 -> 32x32
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),                  # 32 -> 16

            nn.Conv2d(64, 192, kernel_size=5, padding=1),           # 16 -> 16
            nn.BatchNorm2d(192),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),                  # 16 -> 8

            nn.Conv2d(192, 384, kernel_size=5, padding=1),          # 8 -> 8
            nn.BatchNorm2d(384),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),          # 8 -> 8
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),          # 8 -> 8
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),                  # 8 -> 4
        )

        self.head = nn.Sequential(
            nn.Conv2d(256, num_classes, kernel_size=1),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )

    def forward(self, x):
        x = self.features(x)
        return self.head(x)

In [11]:
class DepthwiseSeparableConv2d(nn.Module):
    """
    Depthwise separable convolution:
    - depthwise: (in_ch -> in_ch), groups=in_ch
    - pointwise: 1x1 (in_ch -> out_ch)

    This is a drop-in replacement for a standard Conv2d in terms
    of input/output spatial shape (given same kernel/stride/padding).
    """
    def __init__(self, in_channels, out_channels,
                 kernel_size, stride=1, padding=0, bias=True):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,
            bias=bias,
        )
        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=bias,
        )

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

class DepthwiseAlexNetCIFAR(nn.Module):
    """
    AlexNet-like architecture for CIFAR, but all spatial convolutions
    in the feature extractor are depthwise separable.

    Same topology / channels as AlexNetCIFAR, so you can directly
    compare parameter count and speed.
    """
    def __init__(self, num_classes: int = 10, dropout: float = 0.5):
        super().__init__()
        self.features = nn.Sequential(
            DepthwiseSeparableConv2d(3, 64, kernel_size=7, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),                  # 32 -> 16

            DepthwiseSeparableConv2d(64, 192, kernel_size=5, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),                  # 16 -> 8

            DepthwiseSeparableConv2d(192, 384, kernel_size=5, padding=1),
            nn.BatchNorm2d(384),
            nn.ReLU(inplace=True),

            DepthwiseSeparableConv2d(384, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            DepthwiseSeparableConv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),                  # 8 -> 4
        )

        self.head = nn.Sequential(
            nn.Conv2d(256, num_classes, kernel_size=1),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )

    def forward(self, x):
        x = self.features(x)
        return self.head(x)

In [32]:
model = AlexNetCIFAR()
alexnet_total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters in AlexNet: {alexnet_total_params}")

Total number of parameters in AlexNet: 3640394


In [7]:
epochs = 10
batch_size = 128
lr = 0.001
weight_decay = 5e-4
num_workers = 2
seed = 42

In [8]:
set_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
train_loader, test_loader = get_cifar10_loaders(batch_size=batch_size, num_workers=num_workers)

100%|██████████| 170M/170M [00:12<00:00, 13.8MB/s]


In [15]:
alexnet_test_accuracy = train_and_evaluate(model_name='alexnet')

Epoch 01/10 | Train Loss 1.4239 Acc 47.83% | Val Loss 1.3435 Acc 53.85% | Best Val Acc 53.85%
Epoch 02/10 | Train Loss 1.0593 Acc 62.23% | Val Loss 1.0026 Acc 64.60% | Best Val Acc 64.60%
Epoch 03/10 | Train Loss 0.9214 Acc 67.74% | Val Loss 1.0579 Acc 63.69% | Best Val Acc 64.60%
Epoch 04/10 | Train Loss 0.8335 Acc 70.66% | Val Loss 0.8693 Acc 70.32% | Best Val Acc 70.32%
Epoch 05/10 | Train Loss 0.7790 Acc 72.95% | Val Loss 0.8067 Acc 72.03% | Best Val Acc 72.03%
Epoch 06/10 | Train Loss 0.7176 Acc 75.13% | Val Loss 0.7150 Acc 75.77% | Best Val Acc 75.77%
Epoch 07/10 | Train Loss 0.6796 Acc 76.57% | Val Loss 0.7032 Acc 76.45% | Best Val Acc 76.45%
Epoch 08/10 | Train Loss 0.6417 Acc 77.81% | Val Loss 0.8040 Acc 73.68% | Best Val Acc 76.45%
Epoch 09/10 | Train Loss 0.6148 Acc 78.76% | Val Loss 0.6259 Acc 78.49% | Best Val Acc 78.49%
Epoch 10/10 | Train Loss 0.5914 Acc 79.51% | Val Loss 0.6344 Acc 78.26% | Best Val Acc 78.49%

Final ALEXNET Test Accuracy: 78.26% (loss 0.6344)


In [16]:
model = DepthwiseAlexNetCIFAR()
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters in depthwise separable AlexNet: {total_params}")

Total number of parameters in AlexNet: 269280


In [12]:
depthwise_alexnet_test_accuracy = train_and_evaluate(model_name='depthwise_separable_alexnet')

Epoch 01/10 | Train Loss 1.4992 Acc 45.55% | Val Loss 1.3163 Acc 52.73% | Best Val Acc 52.73%
Epoch 02/10 | Train Loss 1.2160 Acc 56.46% | Val Loss 1.1173 Acc 60.62% | Best Val Acc 60.62%
Epoch 03/10 | Train Loss 1.0988 Acc 60.90% | Val Loss 1.1049 Acc 61.71% | Best Val Acc 61.71%
Epoch 04/10 | Train Loss 1.0286 Acc 63.71% | Val Loss 1.0062 Acc 64.26% | Best Val Acc 64.26%
Epoch 05/10 | Train Loss 0.9703 Acc 65.75% | Val Loss 0.9420 Acc 67.55% | Best Val Acc 67.55%
Epoch 06/10 | Train Loss 0.9184 Acc 67.55% | Val Loss 1.0458 Acc 63.94% | Best Val Acc 67.55%
Epoch 07/10 | Train Loss 0.8827 Acc 69.04% | Val Loss 1.0090 Acc 65.92% | Best Val Acc 67.55%
Epoch 08/10 | Train Loss 0.8484 Acc 70.06% | Val Loss 0.9101 Acc 68.38% | Best Val Acc 68.38%
Epoch 09/10 | Train Loss 0.8169 Acc 71.55% | Val Loss 0.9363 Acc 67.40% | Best Val Acc 68.38%
Epoch 10/10 | Train Loss 0.7992 Acc 72.02% | Val Loss 0.8531 Acc 71.00% | Best Val Acc 71.00%

Final DEPTHWISE_SEPARABLE_ALEXNET Test Accuracy: 71.00% (lo

In [31]:
model = AlexNetCIFAR()
alexnet_speed = benchmark_forward_pass(model, input_size=(512, 3, 32, 32))

Device: cuda
Input size: (512, 3, 32, 32)
Average forward pass time: 22.6286 ms over 100 iters


In [30]:
model = DepthwiseAlexNetCIFAR()
depthwise_alexnet_speed = benchmark_forward_pass(model, input_size=(512, 3, 32, 32))

Device: cuda
Input size: (512, 3, 32, 32)
Average forward pass time: 11.0457 ms over 100 iters


# Conclusion

Using depthwise separable convolution reduced the number of parameters to just 7% (269k / 3.64M) and sped up the forward pass inference almost 2x. All of this while maintaining a slight accuracy drop of 7%.

This makes Depthwise Separable Convolutions useful in situations where we prefer to sacrafice accuracy for a major improvement in inference speed, memory consumption and computation.